# Compare match-outcome models and probability baselines

This notebook compares L2 logistic regression, XGBoost, random forest, and probability baselines using multiclass log loss with a fixed class order. It also saves calibration outputs and the fitted models used by the simulation.

The target is the **90-minute result**:

- `H`: home/listed Team A win after 90 minutes;
- `D`: draw after 90 minutes;
- `A`: away/listed Team B win after 90 minutes.

The Elo ratings are imported from the previous notebook and must be **pre-match** ratings.

## Model comparison metric

The main metric is **multiclass log loss**. This is the right primary metric because the World Cup simulation needs good probabilities, not just the most likely class.

AUC is useful in binary classification, but it is not ideal here:

- the problem has three classes;
- AUC measures ranking ability, not probability calibration;
- two models can have similar AUC but very different simulated tournament probabilities.

We still calculate one-vs-rest multiclass AUC as a secondary diagnostic, but model selection should be based primarily on:

1. multiclass log loss;
2. multiclass Brier score;
3. calibration/reliability;
4. accuracy only as a descriptive statistic.

In [ ]:
from pathlib import Path
import json
import joblib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

def find_project_root():
    """Locate the repo when Jupyter starts in its root or a subfolder."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "01 Input").is_dir() and (candidate / "02 Scripts").is_dir():
            return candidate
    raise FileNotFoundError("Start Jupyter from the project folder or 02 Scripts")


project_root = find_project_root()

input = project_root / "01 Input"
temp = project_root / "03 Temp"
output = project_root / "04 Output"

for folder in [temp, output]:
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Load the pre-match Elo history

This file is produced by the Elo notebook. The fields `home_elo_pre` and `away_elo_pre` are the ratings known before the match.

In [ ]:
elo_path = temp / "match_level_elo_history.parquet"

matches = pd.read_parquet(elo_path)
matches["date"] = pd.to_datetime(matches["date"])

print(f"Loaded {len(matches):,} matches with Elo ratings")
print(matches["date"].min().date(), "to", matches["date"].max().date())

matches.head()

## 2. Reconstruct the 90-minute result

`results.csv` reports scores including extra time but excluding shootout penalties. That score is correct for Elo, but the forecasting target should be the 90-minute result.

Where the goal events reconcile to the reported score, goals through minute 90 define the regulation-time score. The source stores minutes as numbers rather than strings such as `90+3`, so rare stoppage-time values above 90 cannot be distinguished perfectly from extra time; this is a documented approximation. If an incomplete record has a drawn final score, the 90-minute target is still known to be a draw. Other incomplete records use the reported result and retain a method flag.

In [ ]:
goalscorers = pd.read_csv(
    input / "goalscorers.csv",
    parse_dates=["date"],
)

match_key = ["date", "home_team", "away_team"]

goals = goalscorers.copy()
goals["minute"] = pd.to_numeric(goals["minute"], errors="coerce")

# The `team` column is the team credited with the goal, including own goals.
goals["home_goal"] = goals["team"].eq(goals["home_team"]).astype(int)
goals["away_goal"] = goals["team"].eq(goals["away_team"]).astype(int)

# Minutes above 90 are treated as extra-time goals.
goals["home_goal_90"] = (
    goals["team"].eq(goals["home_team"])
    & goals["minute"].notna()
    & goals["minute"].le(90)
).astype(int)

goals["away_goal_90"] = (
    goals["team"].eq(goals["away_team"])
    & goals["minute"].notna()
    & goals["minute"].le(90)
).astype(int)

goals["missing_minute"] = goals["minute"].isna().astype(int)
goals["goal_after_90"] = (
    goals["minute"].notna()
    & goals["minute"].gt(90)
).astype(int)

goal_summary = (
    goals.groupby(match_key, as_index=False)
    .agg(
        event_home_goals=("home_goal", "sum"),
        event_away_goals=("away_goal", "sum"),
        event_home_goals_90=("home_goal_90", "sum"),
        event_away_goals_90=("away_goal_90", "sum"),
        missing_goal_minutes=("missing_minute", "sum"),
        goals_after_90=("goal_after_90", "sum"),
    )
)

matches = matches.merge(
    goal_summary,
    on=match_key,
    how="left",
    validate="one_to_one",
)

count_columns = [
    "event_home_goals",
    "event_away_goals",
    "event_home_goals_90",
    "event_away_goals_90",
    "missing_goal_minutes",
    "goals_after_90",
]

matches[count_columns] = matches[count_columns].fillna(0).astype(int)

In [ ]:
# A complete event record means the goalscorer file reconciles to the reported score.
matches["goal_events_complete"] = (
    matches["event_home_goals"].eq(matches["home_score"])
    & matches["event_away_goals"].eq(matches["away_score"])
    & matches["missing_goal_minutes"].eq(0)
)

# Final 0-0 matches are exact even if there are no goal-event rows.
matches["zero_zero_final"] = (
    matches["home_score"].eq(0)
    & matches["away_score"].eq(0)
)

matches["score_90_from_events"] = (
    matches["goal_events_complete"]
    | matches["zero_zero_final"]
)

matches["home_score_90"] = pd.NA
matches["away_score_90"] = pd.NA

# Regulation-time score reconstructed from goal timings.
from_events = matches["score_90_from_events"]
matches.loc[from_events, "home_score_90"] = matches.loc[from_events, "event_home_goals_90"]
matches.loc[from_events, "away_score_90"] = matches.loc[from_events, "event_away_goals_90"]

# If an incomplete final score is a draw, the target class is known. Use the
# reported drawn score as the best available approximation for form features.
final_draw = matches["home_score"].eq(matches["away_score"])
matches.loc[~from_events & final_draw, "home_score_90"] = matches.loc[
    ~from_events & final_draw, "home_score"
]
matches.loc[~from_events & final_draw, "away_score_90"] = matches.loc[
    ~from_events & final_draw, "away_score"
]

# Remaining incomplete cases: assume the reported result occurred in 90 minutes.
# This is imperfect but should affect a small number of extra-time knockout wins.
assumed = matches["home_score_90"].isna()
matches.loc[assumed, "home_score_90"] = matches.loc[assumed, "home_score"]
matches.loc[assumed, "away_score_90"] = matches.loc[assumed, "away_score"]

matches["home_score_90"] = matches["home_score_90"].astype(int)
matches["away_score_90"] = matches["away_score_90"].astype(int)

matches["score_90_method"] = np.select(
    [
        matches["score_90_from_events"],
        ~matches["score_90_from_events"] & final_draw,
        assumed,
    ],
    [
        "goal_events",
        "draw_target_known",
        "final_result_assumed",
    ],
    default="unknown",
)

matches["result_90"] = np.select(
    [
        matches["home_score_90"].gt(matches["away_score_90"]),
        matches["home_score_90"].lt(matches["away_score_90"]),
    ],
    ["H", "A"],
    default="D",
)

matches["result_90"] = pd.Categorical(
    matches["result_90"],
    categories=["H", "D", "A"],
    ordered=False,
)

summary = (
    matches["score_90_method"]
    .value_counts()
    .rename_axis("score_90_method")
    .reset_index(name="matches")
)
summary["share"] = summary["matches"] / summary["matches"].sum()
summary

## 3. Add recent-form features

These are calculated from earlier matches only. The rolling calculation shifts by one match, and same-day values are reset to the first value of that team-date so unknown kickoff ordering cannot leak one result into another.

In [ ]:
# Use the reconstructed score where available and the reported score otherwise.
# This avoids turning incomplete drawn matches into artificial 0-0 results.
home_rows = matches[[
    "match_id", "date", "home_team", "away_team", "home_score_90", "away_score_90", "home_elo_change"
]].rename(columns={
    "home_team": "team",
    "away_team": "opponent",
    "home_score_90": "goals_for",
    "away_score_90": "goals_against",
    "home_elo_change": "elo_change",
})

away_rows = matches[[
    "match_id", "date", "away_team", "home_team", "away_score_90", "home_score_90", "away_elo_change"
]].rename(columns={
    "away_team": "team",
    "home_team": "opponent",
    "away_score_90": "goals_for",
    "home_score_90": "goals_against",
    "away_elo_change": "elo_change",
})

team_matches = pd.concat([home_rows, away_rows], ignore_index=True)
team_matches = team_matches.sort_values(["team", "date", "match_id"])

team_matches["goal_diff"] = (
    team_matches["goals_for"]
    - team_matches["goals_against"]
)

team_matches["points"] = np.select(
    [
        team_matches["goal_diff"].gt(0),
        team_matches["goal_diff"].eq(0),
    ],
    [3, 1],
    default=0,
)

def rolling_previous_mean(values, window):
    return values.shift(1).rolling(window, min_periods=3).mean()


def rolling_previous_sum(values, window):
    return values.shift(1).rolling(window, min_periods=3).sum()


for window in [5, 10]:
    grouped = team_matches.groupby("team", group_keys=False)

    team_matches[f"goals_for_{window}"] = grouped["goals_for"].transform(
        lambda values: rolling_previous_mean(values, window)
    )

    team_matches[f"goals_against_{window}"] = grouped["goals_against"].transform(
        lambda values: rolling_previous_mean(values, window)
    )

    team_matches[f"goal_diff_{window}"] = grouped["goal_diff"].transform(
        lambda values: rolling_previous_mean(values, window)
    )

    team_matches[f"points_{window}"] = grouped["points"].transform(
        lambda values: rolling_previous_mean(values, window)
    )

    team_matches[f"elo_change_{window}"] = grouped["elo_change"].transform(
        lambda values: rolling_previous_sum(values, window)
    )

form_columns = [
    "goals_for_5", "goals_against_5", "goal_diff_5", "points_5", "elo_change_5",
    "goals_for_10", "goals_against_10", "goal_diff_10", "points_10", "elo_change_10",
]

# Kickoff times are unavailable, so every match on the same date must use
# the same form values based only on earlier dates.
team_matches[form_columns] = (
    team_matches.groupby(["team", "date"])[form_columns]
    .transform("first")
)

In [ ]:
home_form = team_matches[["match_id", "team"] + form_columns].rename(
    columns={
        "team": "home_team_check",
        **{column: f"home_{column}" for column in form_columns},
    }
)

away_form = team_matches[["match_id", "team"] + form_columns].rename(
    columns={
        "team": "away_team_check",
        **{column: f"away_{column}" for column in form_columns},
    }
)

# Merge the home team's form features onto the match row.
model_data = matches.merge(
    home_form,
    left_on=["match_id", "home_team"],
    right_on=["match_id", "home_team_check"],
    how="left",
    validate="one_to_one",
)

# Merge the away team's form features onto the same match row.
model_data = model_data.merge(
    away_form,
    left_on=["match_id", "away_team"],
    right_on=["match_id", "away_team_check"],
    how="left",
    validate="one_to_one",
)

# Sanity checks: each side should have matched to its own team-match row.
assert model_data["home_team"].eq(model_data["home_team_check"]).all()
assert model_data["away_team"].eq(model_data["away_team_check"]).all()

model_data = model_data.drop(columns=["home_team_check", "away_team_check"])

# Add the remaining derived Elo features used below.
model_data["abs_elo_diff_pre"] = (
    model_data["elo_diff_pre"]
    .abs()
)

model_data["mean_elo_pre"] = (
    model_data["home_elo_pre"]
    + model_data["away_elo_pre"]
) / 2

## 4. Define the validation sample

The model sample begins in 2000 after a long Elo warm-up. Provisional teams are dropped, so both teams must have at least 30 previous matches before the current match.

In [ ]:
model_start_date = pd.Timestamp("2000-01-01")
final_training_cutoff = pd.Timestamp("2026-06-10")

model_data = model_data.loc[
    model_data["date"].between(model_start_date, final_training_cutoff)
    & ~model_data["home_provisional_pre"]
    & ~model_data["away_provisional_pre"]
].copy()

model_data["is_friendly"] = model_data["tournament"].eq("Friendly").astype(int)
model_data["is_world_cup"] = model_data["tournament"].eq("FIFA World Cup").astype(int)
model_data["neutral"] = model_data["neutral"].astype(bool)

print(f"Model observations: {len(model_data):,}")
display(model_data["result_90"].value_counts(normalize=True).rename("share"))

score_reconciliation = (
    model_data["score_90_method"]
    .value_counts()
    .rename_axis("score_90_method")
    .reset_index(name="matches")
)
score_reconciliation["share"] = (
    score_reconciliation["matches"] / len(model_data)
)
display(score_reconciliation)

score_changed = (
    model_data["home_score_90"].ne(model_data["home_score"])
    | model_data["away_score_90"].ne(model_data["away_score"])
)
print(f"Reported scores adjusted to 90 minutes: {score_changed.sum():,}")
display(
    model_data.loc[
        score_changed,
        [
            "date", "home_team", "away_team", "home_score", "away_score",
            "home_score_90", "away_score_90", "tournament", "score_90_method",
        ],
    ].head(20)
)

## 5. Feature specifications

Compare two feature sets: Elo and match context with five-match form, and the same features with both five- and ten-match form. This focused comparison tests whether longer form history helps without substantially increasing the runtime.

In [ ]:
feature_specs = {
    "elo_context": {
        "numeric": [
            "home_elo_pre",
            "away_elo_pre",
            "elo_diff_pre",
            "abs_elo_diff_pre",
            "mean_elo_pre",
            "k_factor",
        ],
        "categorical": [
            "neutral",
            "is_friendly",
            "is_world_cup",
        ],
    },

    "elo_context_form_5": {
        "numeric": [
            "home_elo_pre",
            "away_elo_pre",
            "elo_diff_pre",
            "abs_elo_diff_pre",
            "mean_elo_pre",
            "k_factor",

            "home_goals_for_5",
            "home_goals_against_5",
            "home_goal_diff_5",
            "home_points_5",
            "home_elo_change_5",

            "away_goals_for_5",
            "away_goals_against_5",
            "away_goal_diff_5",
            "away_points_5",
            "away_elo_change_5",
        ],
        "categorical": [
            "neutral",
            "is_friendly",
            "is_world_cup",
        ],
    },

    "elo_context_form_5_10": {
        "numeric": [
            "home_elo_pre",
            "away_elo_pre",
            "elo_diff_pre",
            "abs_elo_diff_pre",
            "mean_elo_pre",
            "k_factor",

            "home_goals_for_5",
            "home_goals_against_5",
            "home_goal_diff_5",
            "home_points_5",
            "home_elo_change_5",

            "away_goals_for_5",
            "away_goals_against_5",
            "away_goal_diff_5",
            "away_points_5",
            "away_elo_change_5",

            "home_goals_for_10",
            "home_goals_against_10",
            "home_goal_diff_10",
            "home_points_10",
            "home_elo_change_10",

            "away_goals_for_10",
            "away_goals_against_10",
            "away_goal_diff_10",
            "away_points_10",
            "away_elo_change_10",
        ],
        "categorical": [
            "neutral",
            "is_friendly",
            "is_world_cup",
        ],
    },
}

feature_specs = {
    name: feature_specs[name]
    for name in ["elo_context_form_5", "elo_context_form_5_10"]
}


## 6. Model definitions, metrics and hyperparameter grids

Log loss and Brier score use the full probability vector in the fixed class order `H`, `D`, `A`. Accuracy and AUC are secondary diagnostics. The grids below deliberately keep the search small: two settings per model class, evaluated with both feature sets across 12 validation years. This is a focused comparison, not exhaustive tuning.

In [ ]:
# Fixed class order used throughout the project:
# H = home win, D = draw, A = away win.
classes = ["H", "D", "A"]


def outcome_indices(y_true):
    """Encode outcomes using the fixed H/D/A class order."""

    indices = pd.Categorical(y_true, categories=classes).codes
    if (indices < 0).any():
        raise ValueError("Outcomes contain an unknown class")
    return indices


def one_hot_outcomes(y_true):
    return np.eye(len(classes))[outcome_indices(y_true)]


def multiclass_log_loss(y_true, probabilities):
    """Log loss with probability columns explicitly ordered H/D/A."""

    row = np.arange(len(probabilities))
    observed_probability = probabilities[row, outcome_indices(y_true)]
    return -np.log(np.clip(observed_probability, 1e-15, 1)).mean()


def multiclass_brier_score(y_true, probabilities):
    """Mean squared error between probabilities and one-hot outcomes."""

    return np.mean(
        np.sum((probabilities - one_hot_outcomes(y_true)) ** 2, axis=1)
    )


def make_preprocessor(numeric_features, categorical_features):
    """Create preprocessing for numeric and categorical features."""

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    transformers = []

    if numeric_features:
        transformers.append(
            ("num", numeric_pipeline, numeric_features)
        )

    if categorical_features:
        transformers.append(
            ("cat", categorical_pipeline, categorical_features)
        )

    return ColumnTransformer(transformers=transformers)


def params_to_key(params):
    """Store parameter dictionaries consistently in output tables."""

    return json.dumps(params, sort_keys=True)


def key_to_params(params_key):
    """Recover a parameter dictionary from the stored string version."""

    return json.loads(params_key)


def make_estimator(model_name, params):
    """Create a fresh model using the supplied hyperparameters."""

    if model_name == "logit":
        return LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=1000,
            random_state=2026,
            **params,
        )

    if model_name == "xgboost":
        return XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            num_class=len(classes),
            n_estimators=300,
            tree_method="hist",
            n_jobs=-1,
            random_state=2026,
            **params,
        )

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=600,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=2026,
            **params,
        )

    raise ValueError(f"Unknown model name: {model_name}")


def get_probabilities(fitted_model, x_valid_prepared):
    """Return probabilities in the fixed H/D/A class order."""

    probabilities = fitted_model.predict_proba(x_valid_prepared)
    expected_classes = np.arange(len(classes))

    if not np.array_equal(fitted_model.classes_, expected_classes):
        raise ValueError("Model probability columns are not ordered H/D/A")
    if not np.allclose(probabilities.sum(axis=1), 1):
        raise ValueError("Predicted probabilities do not sum to one")

    return probabilities


def get_predicted_labels(probabilities):
    """Return predicted H/D/A labels."""

    return np.array(classes)[np.argmax(probabilities, axis=1)]


def auc_ovr_macro(y_true, probabilities):
    """One-vs-rest multiclass AUC, used only as a diagnostic."""

    return roc_auc_score(
        one_hot_outcomes(y_true),
        probabilities,
        average="macro",
    )


In [ ]:
# A compact search limits runtime across the temporal validation folds.

model_param_grids = {
    "logit": list(
        ParameterGrid(
            {
                # C controls the overall strength of regularisation.
                # Smaller C = stronger shrinkage.
                "C": [
                    0.1,
                    1.0,
                ],

            }
        )
    ),

    "xgboost": list(
        ParameterGrid(
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03],
                "subsample": [0.8],
                "colsample_bytree": [0.8],
            }
        )
    ),

    "random_forest": list(
        ParameterGrid(
            {
                "max_depth": [None],
                "max_features": ["sqrt", 0.7],
                "min_samples_leaf": [25],
            }
        )
    ),
}
for model_name, param_grid in model_param_grids.items():
    print(
        f"{model_name}: "
        f"{len(param_grid)} parameter combinations"
    )


## 7. Expanding-window temporal validation

Each fold trains on all matches before the validation year and validates on that calendar year. No future matches enter the training set.

In [ ]:
validation_years = list(range(2014, 2026))

folds = []

for year in validation_years:
    train_mask = model_data["date"] < pd.Timestamp(f"{year}-01-01")
    valid_mask = model_data["date"].between(
        pd.Timestamp(f"{year}-01-01"),
        pd.Timestamp(f"{year}-12-31"),
    )

    folds.append(
        {
            "validation_year": year,
            "train_mask": train_mask,
            "valid_mask": valid_mask,
        }
    )

print(f"Validation folds: {len(folds)}")
print([fold["validation_year"] for fold in folds])

In [ ]:
def ranked_probability_score(y_true, probabilities):
    """Ranked probability score for ordered outcomes: A < D < H."""

    ordered_columns = [2, 1, 0]
    ordered_probabilities = probabilities[:, ordered_columns]
    observed = one_hot_outcomes(y_true)[:, ordered_columns]

    cumulative_probabilities = np.cumsum(
        ordered_probabilities,
        axis=1,
    )[:, :-1]

    cumulative_observed = np.cumsum(
        observed,
        axis=1,
    )[:, :-1]

    return np.mean(
        np.sum(
            (
                cumulative_probabilities
                - cumulative_observed
            ) ** 2,
            axis=1,
        )
        / (len(classes) - 1)
    )


# Count model fits so progress is visible while the cell runs.
total_fits = sum(
    len(param_grid) * len(folds)
    for param_grid in model_param_grids.values()
    for _ in feature_specs
)

fit_number = 0
metric_rows = []

for spec_name, spec in feature_specs.items():

    feature_columns = (
        spec["numeric"]
        + spec["categorical"]
    )

    print(f"\nFeature specification: {spec_name}")

    for fold in folds:

        year = fold["validation_year"]

        train = model_data.loc[
            fold["train_mask"]
        ].copy()

        valid = model_data.loc[
            fold["valid_mask"]
        ].copy()

        x_train = train[feature_columns]
        y_train = train["result_90"].astype(str)
        y_train_encoded = outcome_indices(y_train)

        x_valid = valid[feature_columns]
        y_valid = valid["result_90"].astype(str)

        # The preprocessor only depends on the feature set and train fold,
        # so it should be fitted once per spec/fold rather than once per model.
        preprocessor = make_preprocessor(
            spec["numeric"],
            spec["categorical"],
        )

        x_train_prepared = preprocessor.fit_transform(x_train)
        x_valid_prepared = preprocessor.transform(x_valid)

        uniform_probabilities = np.full((len(valid), len(classes)), 1 / len(classes))
        uniform_log_loss = multiclass_log_loss(y_valid, uniform_probabilities)

        for model_name, param_grid in model_param_grids.items():

            for params in param_grid:

                fit_number += 1

                if fit_number == 1 or fit_number % 25 == 0:
                    print(
                        f"Fit {fit_number:,} of {total_fits:,}: "
                        f"{model_name}, {spec_name}, {year}"
                    )

                params_key = params_to_key(params)

                fitted_model = make_estimator(
                    model_name,
                    params,
                )

                fitted_model.fit(x_train_prepared, y_train_encoded)

                probabilities = get_probabilities(
                    fitted_model,
                    x_valid_prepared,
                )

                predicted_labels = get_predicted_labels(probabilities)
                model_log_loss = multiclass_log_loss(y_valid, probabilities)

                metric_rows.append(
                    {
                        "specification": spec_name,
                        "model": model_name,
                        "params_key": params_key,
                        "validation_year": year,
                        "n_train": len(train),
                        "n_valid": len(valid),

                        # Main probability metric.
                        "log_loss": model_log_loss,
                        "uniform_log_loss": uniform_log_loss,
                        "beats_uniform_log_loss": (
                            model_log_loss < uniform_log_loss
                        ),

                        # Supporting probability metrics.
                        "brier_score": multiclass_brier_score(
                            y_valid,
                            probabilities,
                        ),
                        "ranked_probability_score": ranked_probability_score(
                            y_valid,
                            probabilities,
                        ),

                        # Classification / ranking diagnostics.
                        "accuracy": accuracy_score(
                            y_valid,
                            predicted_labels,
                        ),
                        "auc_ovr_macro": auc_ovr_macro(
                            y_valid,
                            probabilities,
                        ),
                    }
                )

metrics = pd.DataFrame(metric_rows)

metrics.to_csv(
    output / "cross_model_metrics_by_fold.csv",
    index=False,
)

print("\nValidation complete.")
display(metrics.head())

## 8. Probability baselines

The substantive models are compared with four progressively more informative references:

- `uniform`: 1/3 for each outcome, retained only as a sanity check;
- `historical_frequency`: the H/D/A frequencies in that fold's training data;
- `venue_only`: L2 multinomial logistic regression using only the neutral-venue dummy;
- `elo_only`: L2 multinomial logistic regression using only pre-match Elo difference.

Every baseline is recalculated within each temporal fold. No validation-year outcomes are used to construct its probabilities.

In [ ]:
def baseline_metric_row(
    specification,
    params,
    year,
    train,
    valid,
    probabilities,
    uniform_log_loss,
):
    """Return the standard validation metrics for one baseline and fold."""

    y_valid = valid["result_90"].astype(str)
    predicted_labels = get_predicted_labels(probabilities)
    model_log_loss = multiclass_log_loss(y_valid, probabilities)

    return {
        "specification": specification,
        "model": "baseline",
        "params_key": params_to_key(params),
        "validation_year": year,
        "n_train": len(train),
        "n_valid": len(valid),
        "log_loss": model_log_loss,
        "uniform_log_loss": uniform_log_loss,
        "beats_uniform_log_loss": model_log_loss < uniform_log_loss,
        "brier_score": multiclass_brier_score(y_valid, probabilities),
        "ranked_probability_score": ranked_probability_score(
            y_valid,
            probabilities,
        ),
        "accuracy": accuracy_score(y_valid, predicted_labels),
        "auc_ovr_macro": auc_ovr_macro(y_valid, probabilities),
    }


baseline_rows = []
baseline_logit_specs = {
    "venue_only": {"numeric": [], "categorical": ["neutral"]},
    "elo_only": {"numeric": ["elo_diff_pre"], "categorical": []},
}

for fold in folds:
    year = fold["validation_year"]
    train = model_data.loc[fold["train_mask"]].copy()
    valid = model_data.loc[fold["valid_mask"]].copy()

    y_train = train["result_90"].astype(str)
    y_valid = valid["result_90"].astype(str)

    uniform_probabilities = np.full((len(valid), len(classes)), 1 / len(classes))
    uniform_log_loss = multiclass_log_loss(y_valid, uniform_probabilities)
    baseline_rows.append(
        baseline_metric_row(
            "uniform",
            {},
            year,
            train,
            valid,
            uniform_probabilities,
            uniform_log_loss,
        )
    )

    historical_probabilities = np.tile(
        one_hot_outcomes(y_train).mean(axis=0),
        (len(valid), 1),
    )
    baseline_rows.append(
        baseline_metric_row(
            "historical_frequency",
            {},
            year,
            train,
            valid,
            historical_probabilities,
            uniform_log_loss,
        )
    )

    for baseline_name, baseline_spec in baseline_logit_specs.items():
        baseline_features = (
            baseline_spec["numeric"] + baseline_spec["categorical"]
        )
        baseline_preprocessor = make_preprocessor(
            baseline_spec["numeric"],
            baseline_spec["categorical"],
        )
        x_train_baseline = train[baseline_features].copy()
        x_valid_baseline = valid[baseline_features].copy()

        # SimpleImputer cannot process a standalone Boolean dataframe,
        # so represent categorical flags as 0/1.
        for feature in baseline_spec["categorical"]:
            x_train_baseline[feature] = x_train_baseline[feature].astype(int)
            x_valid_baseline[feature] = x_valid_baseline[feature].astype(int)

        x_train_prepared = baseline_preprocessor.fit_transform(
            x_train_baseline
        )
        x_valid_prepared = baseline_preprocessor.transform(
            x_valid_baseline
        )

        baseline_model = make_estimator("logit", {"C": 1.0})
        baseline_model.fit(x_train_prepared, outcome_indices(y_train))
        baseline_probabilities = get_probabilities(
            baseline_model,
            x_valid_prepared,
        )

        baseline_rows.append(
            baseline_metric_row(
                baseline_name,
                {"C": 1.0},
                year,
                train,
                valid,
                baseline_probabilities,
                uniform_log_loss,
            )
        )

baseline_metrics = pd.DataFrame(baseline_rows)
baseline_metrics.to_csv(
    output / "baseline_metrics_by_fold.csv",
    index=False,
)

display(baseline_metrics.head())

## 9. Compare models and baselines

Lower log loss and lower Brier score are better. Accuracy is included for reference only. Baselines are reported alongside the best substantive models but cannot be selected as the final fitted model.

In [ ]:
model_comparison = (
    metrics
    .groupby(
        [
            "specification",
            "model",
            "params_key",
        ],
        as_index=False,
    )
    .agg(
        folds=("validation_year", "nunique"),
        mean_log_loss=("log_loss", "mean"),
        sd_log_loss=("log_loss", "std"),
        worst_fold_log_loss=("log_loss", "max"),
        mean_brier_score=("brier_score", "mean"),
        mean_ranked_probability_score=("ranked_probability_score", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_auc_ovr_macro=("auc_ovr_macro", "mean"),
        folds_beating_uniform=("beats_uniform_log_loss", "sum"),
    )
    .sort_values(
        ["mean_log_loss", "sd_log_loss", "mean_brier_score"],
        ascending=[True, True, True],
    )
)

model_comparison.to_csv(
    output / "cross_model_comparison.csv",
    index=False,
)

display(model_comparison.head(30))

baseline_comparison = (
    baseline_metrics
    .groupby(
        ["specification", "model", "params_key"],
        as_index=False,
    )
    .agg(
        folds=("validation_year", "nunique"),
        mean_log_loss=("log_loss", "mean"),
        sd_log_loss=("log_loss", "std"),
        worst_fold_log_loss=("log_loss", "max"),
        mean_brier_score=("brier_score", "mean"),
        mean_ranked_probability_score=("ranked_probability_score", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_auc_ovr_macro=("auc_ovr_macro", "mean"),
        folds_beating_uniform=("beats_uniform_log_loss", "sum"),
    )
    .sort_values("mean_log_loss")
)

baseline_comparison.to_csv(
    output / "baseline_comparison.csv",
    index=False,
)

comparison_with_baselines = (
    pd.concat([model_comparison, baseline_comparison], ignore_index=True)
    .sort_values("mean_log_loss")
    .reset_index(drop=True)
)
comparison_with_baselines.to_csv(
    output / "model_and_baseline_comparison.csv",
    index=False,
)

display(baseline_comparison)

# Compare each leading form specification's best configuration year by year.
form_specifications = ["elo_context_form_5", "elo_context_form_5_10"]
top_form_configs = (
    model_comparison.loc[model_comparison["specification"].isin(form_specifications)]
    .drop_duplicates("specification")
    [["specification", "model", "params_key"]]
)
top_fold_comparison = (
    metrics.merge(
        top_form_configs,
        on=["specification", "model", "params_key"],
    )
    .pivot(index="validation_year", columns="specification", values="log_loss")
)
top_fold_comparison["form_5_minus_form_5_10"] = (
    top_fold_comparison["elo_context_form_5"]
    - top_fold_comparison["elo_context_form_5_10"]
)
display(top_fold_comparison)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

best_model_classes = (
    model_comparison
    .sort_values("mean_log_loss")
    .drop_duplicates("model")
)
plot_data = pd.concat(
    [best_model_classes, baseline_comparison],
    ignore_index=True,
)
plot_data["label"] = np.where(
    plot_data["model"].eq("baseline"),
    "Baseline / " + plot_data["specification"],
    plot_data["model"] + " / " + plot_data["specification"],
)

plot_data = plot_data.sort_values(
    "mean_log_loss",
    ascending=True,
)

ax.barh(
    plot_data["label"],
    plot_data["mean_log_loss"],
)
ax.invert_yaxis()
ax.set_xlabel("Mean validation log loss")
ax.set_title("Best model classes and probability baselines")

plt.tight_layout()
fig.savefig(
    output / "model_and_baseline_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 10. Calibration check for the best model

This checks each outcome probability separately. A well-calibrated model should lie close to the 45-degree line for home wins, draws and away wins.

In [ ]:
# Treat log-loss differences below 0.0005 as practically tied. Within that
# set, prefer the simpler model class and then fewer features.
selection_tolerance = 0.0005
competitive_models = model_comparison.loc[
    model_comparison["mean_log_loss"]
    <= model_comparison["mean_log_loss"].min() + selection_tolerance
].copy()
competitive_models["n_features"] = competitive_models["specification"].map(
    lambda name: len(feature_specs[name]["numeric"] + feature_specs[name]["categorical"])
)
model_complexity = {"logit": 0, "xgboost": 1, "random_forest": 2}
competitive_models["model_complexity"] = competitive_models["model"].map(
    model_complexity
)
best_row = competitive_models.sort_values(
    ["model_complexity", "n_features", "mean_log_loss"]
).iloc[0]

best_spec = best_row["specification"]
best_model = best_row["model"]
best_params = key_to_params(best_row["params_key"])

print("Selected model:")
print(best_model, "/", best_spec)
print(best_params)

prediction_rows = []

selected_features = (
    feature_specs[best_spec]["numeric"]
    + feature_specs[best_spec]["categorical"]
)

for fold in folds:

    year = fold["validation_year"]

    train = model_data.loc[
        fold["train_mask"]
    ].copy()

    valid = model_data.loc[
        fold["valid_mask"]
    ].copy()

    x_train = train[selected_features]
    y_train = train["result_90"].astype(str)
    x_valid = valid[selected_features]

    preprocessor = make_preprocessor(
        feature_specs[best_spec]["numeric"],
        feature_specs[best_spec]["categorical"],
    )

    x_train_prepared = preprocessor.fit_transform(x_train)
    x_valid_prepared = preprocessor.transform(x_valid)

    fitted_model = make_estimator(
        best_model,
        best_params,
    )

    fitted_model.fit(x_train_prepared, outcome_indices(y_train))

    probabilities = get_probabilities(
        fitted_model,
        x_valid_prepared,
    )

    predicted_labels = get_predicted_labels(probabilities)

    fold_predictions = valid[
        [
            "match_id",
            "date",
            "home_team",
            "away_team",
            "result_90",
        ]
    ].copy()

    fold_predictions["specification"] = best_spec
    fold_predictions["model"] = best_model
    fold_predictions["params_key"] = params_to_key(best_params)
    fold_predictions["validation_year"] = year
    fold_predictions["p_home_win"] = probabilities[:, 0]
    fold_predictions["p_draw"] = probabilities[:, 1]
    fold_predictions["p_away_win"] = probabilities[:, 2]
    fold_predictions["predicted_result"] = predicted_labels

    prediction_rows.append(fold_predictions)

validation_predictions = pd.concat(
    prediction_rows,
    ignore_index=True,
)

validation_predictions.to_parquet(
    output / "selected_validation_predictions.parquet",
    index=False,
)

best_predictions = validation_predictions.copy()
probability_columns = dict(zip(classes, ["p_home_win", "p_draw", "p_away_win"]))
calibration_rows = []

for result_label, probability_column in probability_columns.items():
    class_predictions = best_predictions[["match_id", "result_90", probability_column]].copy()
    class_predictions["observed"] = (
        class_predictions["result_90"].astype(str).eq(result_label).astype(int)
    )
    class_predictions["probability_bin"] = pd.cut(
        class_predictions[probability_column],
        bins=np.linspace(0, 1, 11),
        include_lowest=True,
    )

    class_calibration = (
        class_predictions.groupby("probability_bin", observed=True)
        .agg(
            mean_probability=(probability_column, "mean"),
            observed_rate=("observed", "mean"),
            matches=("match_id", "size"),
        )
        .reset_index()
    )
    class_calibration["result"] = result_label
    calibration_rows.append(class_calibration)

calibration = pd.concat(calibration_rows, ignore_index=True)
calibration.to_csv(
    output / "selected_model_calibration.csv",
    index=False,
)

display(calibration)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.plot([0, 1], [0, 1], linestyle="--")
for result_label, class_calibration in calibration.groupby("result"):
    ax.plot(
        class_calibration["mean_probability"],
        class_calibration["observed_rate"],
        marker="o",
        label=result_label,
    )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed outcome rate")
ax.set_title(f"Calibration: {best_model} / {best_spec}")
ax.legend(title="Result")

plt.tight_layout()
fig.savefig(
    output / "selected_model_calibration.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Confusion matrix for the best model

This is descriptive. It should not be used as the main selection criterion.

In [ ]:
confusion = confusion_matrix(
    best_predictions["result_90"].astype(str),
    best_predictions["predicted_result"].astype(str),
    labels=classes,
)

confusion_table = pd.DataFrame(
    confusion,
    index=[f"Actual {label}" for label in classes],
    columns=[f"Predicted {label}" for label in classes],
)

display(confusion_table)

## 12. Fit the selected model on all available pre-tournament data

This final object is what the simulation notebook will use. It is trained on all available matches through `final_training_cutoff`.

In [ ]:
selected_spec = best_spec
selected_model_name = best_model
selected_params = best_params

selected_features = (
    feature_specs[selected_spec]["numeric"]
    + feature_specs[selected_spec]["categorical"]
)

x_full = model_data[selected_features]
y_full = model_data["result_90"].astype(str)
y_full_encoded = outcome_indices(y_full)

final_preprocessor = make_preprocessor(
    feature_specs[selected_spec]["numeric"],
    feature_specs[selected_spec]["categorical"],
)

x_full_prepared = final_preprocessor.fit_transform(x_full)

final_model = make_estimator(
    selected_model_name,
    selected_params,
)

final_model.fit(x_full_prepared, y_full_encoded)

model_bundle = {
    "model": final_model,
    "preprocessor": final_preprocessor,
    "classes": classes,
    "feature_columns": selected_features,
    "numeric_features": feature_specs[selected_spec]["numeric"],
    "categorical_features": feature_specs[selected_spec]["categorical"],
    "specification": selected_spec,
    "model_name": selected_model_name,
    "model_params": selected_params,
    "final_training_cutoff": final_training_cutoff,
}

joblib.dump(
    model_bundle,
    output / "selected_match_outcome_model.joblib",
)

print("Saved selected model:")
print(selected_model_name, "/", selected_spec)
print(selected_params)
print(output / "selected_match_outcome_model.joblib")


## 13. Export the best L2 and XGBoost models separately

The tournament notebook can switch model class by changing one setting. These bundles use the lowest-log-loss configuration within each requested class and are fitted on the same complete pre-tournament sample as the selected overall model.

In [ ]:
best_per_model_class = (
    model_comparison
    .sort_values(["mean_log_loss", "sd_log_loss", "mean_brier_score"])
    .drop_duplicates("model")
    .set_index("model")
)

for model_name in ["logit", "xgboost"]:
    row = best_per_model_class.loc[model_name]
    specification = row["specification"]
    params = key_to_params(row["params_key"])
    features = (
        feature_specs[specification]["numeric"]
        + feature_specs[specification]["categorical"]
    )

    class_preprocessor = make_preprocessor(
        feature_specs[specification]["numeric"],
        feature_specs[specification]["categorical"],
    )
    x_prepared = class_preprocessor.fit_transform(model_data[features])
    class_model = make_estimator(model_name, params)
    class_model.fit(x_prepared, y_full_encoded)

    class_bundle = {
        "model": class_model,
        "preprocessor": class_preprocessor,
        "classes": classes,
        "feature_columns": features,
        "numeric_features": feature_specs[specification]["numeric"],
        "categorical_features": feature_specs[specification]["categorical"],
        "specification": specification,
        "model_name": model_name,
        "model_params": params,
        "validation_mean_log_loss": row["mean_log_loss"],
        "final_training_cutoff": final_training_cutoff,
    }

    model_path = output / f"best_{model_name}_model.joblib"
    joblib.dump(class_bundle, model_path)
    print(f"Saved {model_name}: {specification} -> {model_path}")


## 14. Reminder for knockout simulations

The selected model predicts 90-minute probabilities:

- `p_home_win`
- `p_draw`
- `p_away_win`

For knockout matches, use proportional renormalisation:

\[
P(\text{home advances})=
\frac{p_{home\ win}}
{p_{home\ win}+p_{away\ win}}
\]

This is the probability passed into the bracket simulation.